# Corrected Data Analysis (METR-LA 50-epoch run)

This notebook loads METR-LA data and metadata robustly (avoiding ambiguous  usage), reproduces training/validation plots from the 50-epoch JSON, and shows example sensor time-series using the metadata mapping. Use this notebook when preparing figures for the paper.

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

print('ROOT =', ROOT)
print('DATA =', DATA)
print('RESULTS =', RESULTS)

ROOT = C:\Users\rockk\OneDrive\Desktop\traffic-flow-gnn
DATA = C:\Users\rockk\OneDrive\Desktop\traffic-flow-gnn\data
RESULTS = C:\Users\rockk\OneDrive\Desktop\traffic-flow-gnn\results


In [ ]:
# Load metadata
meta_path = DATA / 'metr-la' / 'METR-LA-META.csv'
if not meta_path.exists():
    meta_path = DATA / 'METR-LA-META.csv'
meta_df = pd.read_csv(meta_path)
# Ensure sensor_id column as string for matching
meta_df['sensor_id'] = meta_df['sensor_id'].astype(str)
print('Loaded metadata rows =', len(meta_df))
meta_df.head()

In [ ]:
# Robustly load METR-LA.csv and align columns to meta sensor ids
data_path = DATA / 'metr-la' / 'METR-LA.csv'
if not data_path.exists():
    data_path = DATA / 'METR-LA.csv'
df = pd.read_csv(data_path, low_memory=False)
print('Raw columns sample:', list(df.columns[:10]))
# Detect index/timestamp column in first column
first_col = str(df.columns[0])
if first_col.lower().startswith('unnamed') or 'time' in first_col.lower() or 'date' in first_col.lower():
    print('Detected index/timestamp column:', first_col)
    df = df.set_index(df.columns[0])

# Normalize columns to strings
df.columns = df.columns.astype(str)
# Align columns to meta: if columns already contain sensor ids, keep them, else remap by position
if set(meta_df['sensor_id'].tolist()).issubset(set(df.columns)):
    print('Data columns already include sensor IDs; no remap needed')
else:
    # Exclude index column if it's part of columns (we already set it to index above)
    data_cols = list(df.columns)
    if len(data_cols) == len(meta_df):
        print('Remapping data columns by position to metadata sensor IDs')
        df.columns = meta_df['sensor_id'].tolist()
    else:
        print('Column count does not match metadata; leaving original column names. Please verify mapping manually if plots look wrong.')

print('Final data shape:', df.shape)
df.iloc[:3, :5]


## Example sensor time-series (first 3 sensors from metadata)

In [ ]:
sample_sensors = meta_df['sensor_id'].astype(str).tolist()[:3]  # first 3 sensors from meta
plt.figure(figsize=(12,4))
for sid in sample_sensors:
    if sid not in df.columns:
        print(f'Sensor {sid} not found in data columns; skipping')
        continue
    series = pd.to_numeric(df[sid], errors='coerce')
    plt.plot(series[:500].values, label=f'Sensor {sid}')
plt.title('Traffic Speed Patterns (first 500 timesteps)')
plt.xlabel('Timestep')
plt.ylabel('Speed')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Reproduce 50-epoch training & test plots (JSON)

In [ ]:
# Load results JSON (50-epoch)
json_path = RESULTS / 'enhanced_training_results_metrla_20260916_165902.json'
if not json_path.exists():
    candidates = sorted(RESULTS.glob('enhanced_training_results_metrla_*.json'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        candidates = sorted(RESULTS.glob('enhanced_training_results_*.json'), key=lambda p: p.stat().st_mtime)
    if candidates:
        json_path = candidates[-1]

print(f'Loading training results from: {json_path.name}')
with open(json_path, 'r', encoding='utf-8') as f:
    res = json.load(f)
history = res.get('training_history', res.get('history', {}))
train_loss = history.get('train_loss', [])
val_loss = history.get('val_loss', [])
train_metrics = history.get('train_metrics', [])
val_metrics = history.get('val_metrics', [])
lr = history.get('learning_rates', [])

# Plot loss and MAE trend
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (METR-LA 50-epoch)')
plt.legend()
plt.grid(alpha=0.2)

if train_metrics and val_metrics:
    train_mae = [m.get('mae', None) for m in train_metrics]
    val_mae = [m.get('mae', None) for m in val_metrics]
    plt.subplot(1,2,2)
    plt.plot(train_mae, label='Train MAE')
    plt.plot(val_mae, label='Val MAE')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.title('MAE trend (METR-LA 50-epoch)')
    plt.legend()
    plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Print test metrics
test_metrics = res.get('test_metrics', res.get('final_test_metrics', {}))
print('Test metrics (METR-LA 50-epoch run):')
for k, v in test_metrics.items():
    print(f'{k}: {v}')